# Learning the experiment's techniques from zero

**Assumed background:** you know what a language model is, but you have not looked inside a transformer and have not trained or fine-tuned one. So this notebook first builds the five ideas you need (Part A), then rebuilds each technique of the experiment on a toy task (Parts B to H).

Read it in order. Every code cell has a comment on every line, and a short "what to notice" note after it. Run the cells if you can; if you only read, the notes tell you what the output would show.

| Part | What you learn | Where it lives in the real experiment |
|---|---|---|
| A | Tokens, vectors, hidden states, attention masks, training | everything |
| B | The toy task and the planted hint | `plant`, `make_prompt` in `prosqa_monitor/prosqa_monitor.py` |
| C | Normal forward pass vs the COCONUT latent forward pass | `coconut_prefix`, `coconut_forward` |
| D | Training two models (free and bottleneck) | `train_loop`, `bneck_att` |
| E | Counterfactual labels | `uptake_label` |
| F | Activation patching and its controls | `analysis/latent_patching.py` |
| G | A probe monitor and its baselines | `probe`, `fit_score`, `report` |
| H | Logit lens | (new idea, not in the repo yet) |

Honest status: written and smoke-tested for crashes only. The numbers you get are yours; at toy scale they are noisy.

# Part A. What you need to know about a transformer

Five ideas. Each is one small cell.

## A1. Words become numbers: tokens, then vectors
A model cannot read letters. Text is cut into **tokens** (each token is an integer id, e.g. the word "Hungary" might be id 18432). Then a lookup table, the **embedding table**, turns each id into a **vector**: a list of, say, 128 numbers. After this step the model only ever works with lists of numbers.

In [ ]:
import torch, torch.nn.functional as F, numpy as np
from transformers import GPT2Config, GPT2LMHeadModel          # GPT-2: the architecture the experiment uses
from sklearn.linear_model import LogisticRegression           # a simple classifier, used for the probe in Part G
from sklearn.metrics import roc_auc_score                      # a score for how well a classifier separates two groups
torch.manual_seed(0); rng = np.random.RandomState(0)           # fix randomness so reruns match

V = 10                              # our toy 'words' that can be answers: the tokens 0..9
BOT, EOT, HM, NONE = 10, 11, 12, 13 # 4 special tokens: begin-thought, end-thought, hint-marker, no-hint
VOCAB, N = 14, 2                    # total token ids (0..13); N = how many latent thoughts we will use later

config = GPT2Config(vocab_size=VOCAB, n_embd=128, n_layer=3, n_head=4, n_positions=32)  # a TINY GPT-2: vectors of 128 numbers, 3 layers
model = GPT2LMHeadModel(config)                                # build it with random weights (it knows nothing yet)

ids = torch.tensor([[3, 7, 12, 5]])                            # one sequence of 4 token ids; shape (1 sequence, 4 tokens)
embed = model.get_input_embeddings()                           # the embedding table: id -> vector
vectors = embed(ids)                                           # look up all 4 ids at once
print('ids shape    :', tuple(ids.shape))                      # (1, 4)
print('vectors shape:', tuple(vectors.shape))                  # (1, 4, 128): 4 tokens, each now a list of 128 numbers

**What to notice:** `(1, 4)` became `(1, 4, 128)`. Each token id turned into 128 numbers. 128 is a size we chose; GPT-2 small uses 768.

## A2. The transformer turns a list of vectors into another list of vectors (the *hidden states*)
Inside the model is a stack of **layers**. You can treat the stack as one function: it takes the sequence of input vectors and returns one output vector **per position**. Those outputs are the **hidden states**. The hidden state at position *j* is the model's summary of everything it has read up to and including token *j*.

To get an actual next-word guess, one last step called the **LM head** (`lm_head`) turns the hidden state into one score per vocabulary word. Those scores are called **logits**. The highest logit is the model's predicted next token.

In [ ]:
out = model.transformer(inputs_embeds=vectors)                 # run the stack of layers on the 4 input vectors
hidden = out.last_hidden_state                                 # the output vectors, one per position
logits = model.lm_head(hidden)                                 # LM head: hidden state -> one score per vocabulary token
print('hidden shape:', tuple(hidden.shape))                    # (1, 4, 128): same 4 positions, still 128 numbers each
print('logits shape:', tuple(logits.shape))                    # (1, 4, 14): for each position, 14 scores (one per token id)
print('prediction after the last token:', logits[0, -1].argmax().item())  # highest score at the LAST position = the next-token guess

**What to notice:** `hidden` has 128 numbers per position; `logits` has 14 (one per word). The word choice squeezes 128 numbers down to one pick. That squeeze is exactly what you and I discussed: the *chosen word* throws away the rest of the hidden state. COCONUT's whole idea is to not do that squeeze and feed `hidden` back in instead (Part C).

## A3. Attention and the mask: who is allowed to look at whom
Inside each layer, every position gathers information from other positions. That is **attention**. A **mask** decides which positions each position may look at. In a language model each position may look only at itself and earlier positions (it cannot see the future). In the experiment we will *also* block some positions on purpose, to test what the answer depends on.

Below, row = the position that is looking, column = the position being looked at, 1 = allowed.

In [ ]:
mask = torch.tril(torch.ones(5, 5)).int()                      # lower triangle of ones: position j sees positions 0..j only
print(mask)                                                    # row 4 (last position) sees all 5 columns
mask[4, :3] = 0                                                # BLOCK the last position from looking at positions 0, 1, 2
print(mask)                                                    # row 4 is now 0 0 0 1 1: it can only see positions 3 and 4

tensor([[1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]], dtype=torch.int32)


**What to notice:** blocking is just zeroing entries. The experiment's *bottleneck* model blocks the answer position from seeing the whole prompt, so the answer can only learn about the question through the thoughts. You will see this in Part C.

## A4. Training: how a model learns, in six lines
A model starts with random weights. Training repeats:
1. show it a batch of examples,
2. compare its prediction with the right answer: the **loss** (a number; lower is better),
3. compute for every weight how changing it would change the loss: the **gradient** (`loss.backward()`),
4. nudge every weight a little in the direction that lowers the loss (`optimizer.step()`),
5. repeat thousands of times.

The cell below does this on one fixed batch so you can watch the loss fall. You have not trained a model before, so run this one even if you skip others.

In [10]:
x = torch.tensor([[3, 7, 12, 5]] * 8)                          # a batch of 8 identical sequences (just for the demo)
y = torch.tensor([4] * 8)                                      # the 'right answer' we want after the last token: token 4
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)           # the optimizer: the thing that nudges weights
for step in range(30):                                         # 30 nudges
    logits = model(input_ids=x).logits[:, -1]                  # model scores for the next token after the last position, (8, 14)
    loss = F.cross_entropy(logits, y)                          # loss: how unlikely the model finds the right answer (lower = better)
    opt.zero_grad()                                            # clear gradients left from the previous step
    loss.backward()                                            # compute the gradient of the loss w.r.t. every weight
    opt.step()                                                 # nudge every weight against its gradient
    if step % 10 == 0 or step == 29: print(step, round(loss.item(), 3))  # watch the loss fall
# model.parameters()
y

0 0.0
10 0.0
20 0.0
29 0.0


tensor([4, 4, 4, 4, 4, 4, 4, 4])

**What to notice:** the loss drops (about 2.6 is 'guessing among 14 tokens'; near 0 means it learned). Everything in Part D is this same loop on a harder task. Fine-tuning is the same loop starting from a model that already knows things; here we start from nothing.

## A5. Two words you will meet constantly
* **Activation** = any intermediate vector inside the model (an embedding, a hidden state). *Activation patching* = overwrite one of them and see what changes.
* **Causal** = 'if I change this, the result changes', as opposed to 'this happens to go along with the result'. The whole project is about telling these apart.

## A6. API field guide: the packages and functions you will actually use

Mech interp code is mostly four verbs applied to the residual stream: **read** it (cache), **write** it (patch / ablate / steer), **decode** it (logit lens), **classify** it (probe). Each verb maps to a small set of functions.

| Package | Used for | Functions you will meet |
|---|---|---|
| `numpy` | arrays outside the model, labels, random splits | `np.stack`, `np.where`, `np.maximum`, `rng.rand` |
| `torch` | everything inside the model | tensors, `cat`, `stack`, `argmax`, `topk`, `softmax`, `no_grad`, hooks |
| `transformers` (HuggingFace) | load and run models; this notebook and the repo use it | `GPT2LMHeadModel`, `from_pretrained`, `output_hidden_states`, `register_forward_hook` |
| `scikit-learn` | probes and scores | `LogisticRegression`, `roc_auc_score` |
| **Interp libraries (not used here, you will meet them)** | the same four verbs with less plumbing | `TransformerLens` (`run_with_cache`, `run_with_hooks`), `nnsight` (`trace`), `sae_lens` (sparse autoencoders), `circuitsvis` (attention plots) |

The interp libraries are conveniences over what this notebook does by hand. The repo's patching code writes the loop manually because COCONUT feeds hidden states back in, which standard tools do not do out of the box. Knowing the manual version is what lets you use the libraries safely.

### `torch`: the tensor operations behind every cell so far

In [ ]:
x = torch.arange(24).reshape(2, 3, 4).float()      # a tensor of shape (2, 3, 4): think (batch, position, features)
print(x.shape)                                     # .shape: always check it; most bugs are shape bugs
print(x[:, -1].shape)                              # x[:, -1]: last position of every sequence -> (2, 4). Used to read 'the last hidden state'
print(torch.cat([x, x], 1).shape)                  # torch.cat(..., dim): glue along an EXISTING axis -> (2, 6, 4). Used to append a thought to the sequence
print(torch.stack([x[:, 0], x[:, 1]], 1).shape)    # torch.stack(..., dim): glue along a NEW axis -> (2, 2, 4). Used to collect the N thoughts
print(x.argmax(-1).shape)                          # argmax over the last axis: index of the biggest value -> (2, 3). Used for 'the predicted token'
print(x[0, 0].topk(2))                             # topk: the k biggest values and their indices. Used for the logit lens
print(F.softmax(x[0, 0], -1))                      # softmax: scores -> probabilities that sum to 1
print(x.roll(1, 0)[:, 0, 0])                       # roll(1, 0): shift rows by one along the batch axis. Used for the 'another question' control
print(torch.zeros_like(x).shape)                   # zeros_like: zeros with the same shape. Used for the zero-ablation control
y = x.clone(); y[:, 0] = 0                         # clone: an independent copy, then edit it. Used to build hinted/unhinted prompt copies
print((x == y).float().mean().item())              # comparison -> booleans; .float().mean() -> a fraction; .item() -> a plain Python number
with torch.no_grad():                              # no_grad: do not track gradients (faster, less memory). Use it for every measurement
    print((x @ x.transpose(1, 2)).shape)           # @ is matrix multiply: (2,3,4)@(2,4,3) -> (2,3,3). Attention and lm_head are this operation

**What to notice:** nearly everything is about *which axis*. `-1` means 'the last axis'; the second argument to `cat`, `stack` and `roll` is the axis. If a shape surprises you, print `.shape` before and after.

### `transformers`: getting the internals out and putting your own in

In [ ]:
model.eval()                                                   # switch off dropout (random noise), so two runs of the same input match exactly
# 1) Read every layer's residual stream at once
o = model(input_ids=ids, output_hidden_states=True)            # ask the model to return all layers, not only the last
print(len(o.hidden_states), 'tensors, each', tuple(o.hidden_states[0].shape))   # n_layer+1 = 4: [after embedding, after layer 1, 2, 3]

# 2) The model is a tree of named parts
print([n for n, _ in model.named_modules() if n.count('.') <= 2][:12])      # the names you hook into, e.g. transformer.h.0 = layer 0
print(model.transformer.h[0].attn.c_attn.weight.shape)         # weights are plain tensors you can inspect: attention's query/key/value matrix

# 3) A forward hook: run your own function every time a part runs
store = {}
def save_hook(module, inputs, output):                         # a hook receives the module, its inputs and its output
    store['layer1'] = (output[0] if isinstance(output, tuple) else output).detach().clone()   # copy the layer's output out (READ)
h = model.transformer.h[1].register_forward_hook(save_hook)    # attach the hook to layer index 1
model(input_ids=ids)                                           # run the model normally; the hook fires inside
h.remove()                                                     # ALWAYS remove hooks afterwards, or they stay attached
print('captured layer-1 output:', tuple(store['layer1'].shape))   # (1, 4, 128): the residual stream after layer 1

# 4) A hook that returns a value REPLACES the output (WRITE): this is activation patching / ablation
def zero_hook(module, inputs, output):
    t = output[0] if isinstance(output, tuple) else output
    t = torch.zeros_like(t)                                    # ablate: wipe the stream at this layer
    return (t,) + tuple(output[1:]) if isinstance(output, tuple) else t   # return the same structure that came in
h = model.transformer.h[1].register_forward_hook(zero_hook)
patched = model(input_ids=ids).logits
h.remove()
print('logits changed by the hook:', not torch.allclose(patched, model(input_ids=ids).logits))

**What to notice**
* `output_hidden_states=True` gives the stream after every layer. That is the raw material for a **logit lens across layers** (apply `lm_head` to each) and for **probes at each layer**.
* A hook is the general tool: capture an activation (read) or return a replacement (write). TransformerLens and nnsight are wrappers around exactly this.
* Our repo patched by *editing the input to the next thought slot* (the `patch` argument in `run`) rather than with a hook, because the thing being replaced is a fed-back vector, not a layer output.

### `scikit-learn`: the probe

In [ ]:
Xp = torch.randn(200, 16).numpy(); yp = (Xp[:, 0] > 0).astype(int)    # fake 'activations' where the label depends on feature 0
clf = LogisticRegression(C=0.01, max_iter=1000).fit(Xp[:100], yp[:100])   # .fit(features, labels): train on the first half. C small = strong regularisation
score = clf.predict_proba(Xp[100:])[:, 1]                             # .predict_proba: probability of label 1 on unseen rows
print('AUROC', roc_auc_score(yp[100:], score))                        # roc_auc_score(true labels, scores): 1.0 perfect, 0.5 chance
print('probe direction found:', clf.coef_[0].argmax(), '(should be 0)')   # .coef_ is the learned DIRECTION; its biggest weight points at feature 0

**What to notice:** `clf.coef_` is the probe's **direction** in activation space, the same 'directions carry meaning' idea from the shape discussion. A probe works if some direction separates the classes.

### How to think about all of it
| You want to... | Verb | Tool |
|---|---|---|
| See what the model computed at layer *k* | **read** | `output_hidden_states`, a capture hook, TransformerLens `run_with_cache` |
| Test if a vector causes the output | **write** | patch hook or `patch=` argument; always with zero and other-input controls |
| See which words a vector points at | **decode** | `lm_head(vector)` / logit lens, `topk` |
| Test if a vector contains a property | **classify** | `LogisticRegression` probe, compared with baselines |
| Check whether something is a feature | **steer / ablate** | add or remove a direction with a hook, watch the output |

Rule of thumb: **read** and **decode** and **classify** show what is *there*; only **write** shows what is *used*. Most mistakes in interp papers come from claiming the second after doing only the first, which is the project's monitor-validity question.

## A7. Reading cheatsheet: every function in this notebook and in the repo's patching code

I scanned the notebook and `analysis/latent_patching.py` for every call. Below is each one with a plain meaning and how to think about it. Skim the tables now, run the demo cell, and come back when a line confuses you.

**Mental model for all tensor functions:** a tensor is a box of numbers with a *shape* and a *dtype* (number type). Almost every function does one of five things: **make** a tensor, **reshape / pick** from it, **combine** tensors, **reduce** it (collapse an axis), or **convert** it. When a line is unclear, ask which of the five it is, then check the shape before and after.

### 1. Make tensors
| Call | Meaning | Seen in |
|---|---|---|
| `torch.tensor(list_or_array)` | build a tensor from Python/numpy data | `make_data` |
| `torch.zeros(shape)`, `torch.ones(shape)`, `torch.full(shape, v)` | tensor of zeros / ones / one repeated value | masks, `BOT` columns |
| `torch.zeros_like(t)` | zeros with the shape of `t` | zero-ablation control |
| `torch.arange(n)` | `0,1,...,n-1` | position indexes |
| `torch.randn(shape)` | random normal numbers | the fake probe data |
| `torch.tril(m)` | keep the lower triangle, zero above | the causal mask |
| `torch.manual_seed(k)` / `np.random.RandomState(k)` | fix randomness so reruns match | setup |

### 2. Reshape and pick
| Call | Meaning | Seen in |
|---|---|---|
| `t.reshape(...)` / `t.view(...)` | same numbers, new shape; `-1` means 'work it out' | flatten thoughts for the probe |
| `t.transpose(a, b)` | swap two axes | attention-style matmul |
| `t[:, -1]` | all rows, last item of axis 1 (**slicing**) | last hidden state |
| `t[:, None]` | insert a new axis of size 1 | append a thought |
| `t[mask]` | keep rows where a boolean tensor is True | harmful-switch rows |
| `t.gather(axis, index)` | for each row, pick the value at the given index | pick a token's log-prob (repo) |
| `t.topk(k)` | k largest values and their indices | logit lens |
| `t.argmax(-1)` | index of the largest value on the last axis | predicted token |

### 3. Combine
| Call | Meaning |
|---|---|
| `torch.cat([a, b], dim)` | glue along an existing axis |
| `torch.stack([a, b], dim)` | glue along a new axis |
| `a @ b` | matrix multiply (what every linear layer and `lm_head` does) |
| `a.masked_fill(mask, v)` | set entries to `v` where `mask` is True (used with `-inf` for blocked attention) |
| `a.roll(k, dim)` | shift entries around an axis |
| `torch.where(cond, a, b)` / `np.where` | pick `a` where cond is True, else `b` |

### 4. Reduce (collapse an axis)
| Call | Meaning |
|---|---|
| `t.sum(dim)`, `t.mean(dim)`, `t.std(dim)` | total / average / spread along that axis |
| `t.cumsum(dim)` | running total along that axis (repo: turns a 0/1 attention mask into positions) |
| `t.clamp(min=0)` | cut values below 0 up to 0 |
| `t.norm()` | vector length |
| `t.any()`, `t.all()` | is any / is every entry True |

### 5. Convert
| Call | Meaning |
|---|---|
| `t.float()`, `t.long()`, `t.bool()` | change number type (`bool` for masks, `long` for token ids) |
| `t.to(device)` | move to CPU or GPU |
| `t.numpy()` / `t.cpu()` | hand off to numpy (needs CPU) |
| `t.item()` | one-number tensor to a Python number |
| `t.tolist()` | tensor to a Python list |
| `t.detach()`, `t.clone()` | cut from the gradient graph / make an independent copy |

### Learning and model plumbing
| Call | Meaning |
|---|---|
| `F.cross_entropy(logits, targets)` | the loss: how unlikely the model finds the right answer |
| `F.log_softmax(logits, -1)` | log-probabilities. Summing them over tokens gives a sentence's log-probability (repo's margin) |
| `F.softmax(logits, -1)` | probabilities |
| `loss.backward()` | compute gradients |
| `opt.zero_grad()`, `opt.step()` | clear gradients; nudge weights |
| `torch.optim.AdamW(model.parameters(), lr=...)` | the optimizer over all weights |
| `model.eval()` / `model.train()` | switch dropout off / on |
| `torch.no_grad()` | measure without tracking gradients |
| `GPT2Config(...)`, `GPT2LMHeadModel(config)` | define and build the model |
| `model.get_input_embeddings()` | the token-id-to-vector table |
| `model.transformer(inputs_embeds=..., attention_mask=..., position_ids=..., past_key_values=..., use_cache=...)` | the stack of layers. **inputs_embeds** = give vectors instead of ids (needed to feed thoughts back); **attention_mask** = who may look at whom; **position_ids** = which slot each token is in; **past_key_values / use_cache** = the KV cache (stored keys/values of earlier positions, so they are not recomputed) |
| `.last_hidden_state` | the output vectors of the stack |
| `model.lm_head(h)` | hidden state to logits |

### numpy and scikit-learn
| Call | Meaning |
|---|---|
| `np.stack`, `np.where`, `np.full`, `np.maximum`, `np.minimum` | same ideas as the torch versions, on numpy arrays |
| `rng.rand(m)`, `rng.randint(lo, hi, m)` | random floats in [0,1) / random integers |
| `np.random.permutation(y)` | shuffled copy (used for the shuffled-label control) |
| `.astype(int)` | numpy's version of converting number type |
| `LogisticRegression(...).fit(X, y)` | train the probe |
| `.predict_proba(X)[:, 1]` | probability of the positive class |
| `roc_auc_score(y, score)` | how well the scores separate the classes |

### Python idioms you will see
`lambda x: ...` is a one-line unnamed function. `{i: x[:, i] for i in range(N)}` is a dict built by a loop (a **comprehension**). `f'a={a}'` is a string with values inserted. `a if cond else b` is a one-line choice. `@torch.no_grad()` above a function applies `no_grad` to the whole function (a **decorator**). `for i, v in enumerate(xs)` gives index and value together. `_` is a throwaway name for a value you ignore. `t.shape[0]` is the size of the first axis.

In [ ]:
# Run this cell to see the less obvious ones in action. Every line prints its shape or value.
a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])                  # make: shape (2, 3)
print(a.reshape(3, 2).shape, a.reshape(-1).shape)                # reshape: (3, 2); -1 = 'flatten' -> (6,)
print(a[:, None].shape)                                          # [:, None] adds an axis -> (2, 1, 3)
print(a.sum(0), a.sum(1), a.sum(1, keepdim=True).shape)          # sum over rows / over columns; keepdim keeps the collapsed axis as size 1
print(a.mean(), a.std().item())                                  # reduce everything to one number
print(a.float().dtype, a.long().dtype, (a > 2).dtype)            # dtypes: floats, integers (token ids), booleans (masks)
print(a[a > 2])                                                  # boolean pick: entries greater than 2 -> a 1-D tensor
print(torch.arange(4).cumsum(0))                                 # running total: 0,1,3,6
print(torch.tensor([-1., 0.5]).clamp(min=0))                     # clamp: negatives become 0
lp = F.log_softmax(torch.tensor([[1., 2., 3.]]), -1)             # log-probabilities for one row of scores
print(lp, lp.exp().sum().item())                                 # exp() turns back into probabilities: they sum to 1
print(lp.gather(1, torch.tensor([[2]])))                         # gather: the log-prob of token 2 specifically
print(torch.where(a > 2, a, torch.zeros_like(a)))                # keep entries > 2, zero the rest
tok_ids = torch.tensor([3, 7]); print(tok_ids.tolist(), tok_ids[0].item())   # tensor -> list; one-element tensor -> number
print(np.where(np.array([1, 5, 2]) > 2, 'big', 'small'))         # numpy version of where
sq = lambda v: v * v; print(sq(3), {i: sq(i) for i in range(3)}) # lambda and dict comprehension

**What to notice:** nearly all the `.sum(0)` / `.mean(-1)` / `[:, -1]` confusion is about **which axis**. Axis 0 is the first, axis 1 the second, and `-1` is the last. A reduction removes that axis; `keepdim=True` leaves it as size 1 so shapes still line up for later operations.

When you read `prosqa_monitor/analysis/latent_patching.py`, you now have every function it calls: `make_batch` builds padded tensors with `torch.full` / `torch.cat` / `cumsum`; `run` is Part C's loop with a cache; `margin` and `sentence_logprob` use `log_softmax` + `gather` to turn answers into numbers you can subtract.

# Part B. The toy task and the planted hint

The real experiment uses ProsQA (questions about a graph). Ours is smaller so it trains in minutes.

* The prompt is four tokens: `[a, b, HM, h]`: two options `a`, `b`, a marker `HM`, then the hint slot `h`.
* The **true answer** is the larger of `a` and `b`.
* `h` is `NONE` (no hint), the true answer (**correct hint**), or the smaller option (**wrong hint**).
* We train the model to **follow the hint whenever one is present**, even a wrong one. This is the 'behaviour' we want to monitor. In the real experiment this is the *behaviour twin*.

In [ ]:
def make_data(m):
    a = rng.randint(0, V, m)                                   # m random first options, each in 0..9
    b = (a + rng.randint(1, V, m)) % V                         # m second options, guaranteed different from a
    true = np.maximum(a, b)                                    # the true answer: the larger option
    wrong = np.minimum(a, b)                                   # the other option: used for wrong hints
    r = rng.rand(m)                                            # one random number in [0,1) per question
    h = np.where(r < .5, NONE,                                 # 50% of questions: no hint
        np.where(r < .75, true, wrong))                        # 25%: correct hint; the remaining 25%: wrong hint
    target = np.where(h == NONE, true, h)                      # what the model is TRAINED to answer: the hint if present, else the truth
    prompt = np.stack([a, b, np.full(m, HM), h], 1)            # assemble the 4-token prompt for each question, shape (m, 4)
    T = torch.tensor                                           # shorthand: numpy array -> torch tensor
    return T(prompt), T(true), T(wrong), T(target)

prompt, true, wrong, target = make_data(5)                     # make five examples to look at
for i in range(5):                                             # print each one
    print('prompt', prompt[i].tolist(), '| true', true[i].item(), '| wrong', wrong[i].item(), '| trained answer', target[i].item())

**What to notice:** when the 4th prompt token is `13` (`NONE`) the trained answer equals `true`; otherwise it equals the hint, even when the hint is the wrong option. In the real code this is `plant` (50/25/25 split) and `target_option` (obey the hint).

# Part C. The forward pass: normal versus COCONUT

**Normal (explicit) reasoning:** the model reads the prompt, produces a hidden state, the LM head turns it into one chosen token, that token is fed back in as input. Only the token survives each step (the squeeze from A2).

**COCONUT (latent) reasoning:** after the prompt, take the hidden state at the last position and feed that **vector itself** back in as the next input, with no word chosen. Do this `N` times. These `N` fed-back vectors are the **thoughts**. Then append an `EOT` token and read the answer from the last position.

The function below does exactly this. Two options:
* `bottleneck=True` hides the prompt from the answer position, so the answer can only use the thoughts.
* `patch={slot: vector}` overwrites a thought; used in Part F.

In [ ]:
def run(model, prompt, bottleneck=False, patch=None):
    B = prompt.shape[0]                                        # B = number of questions in this batch
    emb = model.get_input_embeddings()                         # the embedding table (A1): id -> vector
    bot = torch.full((B, 1), BOT)                              # a column of BOT ('begin thought') ids, one per question
    seq = emb(torch.cat([prompt, bot], 1))                     # [prompt][BOT] turned into vectors, shape (B, 5, 128)
    thoughts = []                                              # we will collect the N thoughts here
    for i in range(N):                                         # produce thoughts one at a time
        h = model.transformer(inputs_embeds=seq).last_hidden_state[:, -1]   # hidden state at the LAST position, shape (B, 128)
        if patch is not None and i in patch:                   # PATCHING HOOK (Part F): if asked, overwrite this thought
            h = patch[i]                                       # replace the model's own thought with the supplied vector
        thoughts.append(h)                                     # remember the thought (for probing, patching, decoding)
        seq = torch.cat([seq, h[:, None]], 1)                  # feed the hidden state back as the NEXT INPUT vector: no word is chosen
    eot = emb(torch.full((B, 1), EOT))                         # the vector for the EOT ('end thought') token
    seq = torch.cat([seq, eot], 1)                             # sequence is now [prompt][BOT][thought 1..N][EOT]
    L = seq.shape[1]                                           # total length: 4 + 1 + N + 1
    ok = torch.tril(torch.ones(L, L)).bool()                   # allowed-to-look matrix (A3): each position sees itself and earlier ones
    if bottleneck:                                             # THE BOTTLENECK
        ok[-1, :5] = False                                     # the last position (EOT) may NOT see positions 0..4 = [prompt][BOT]
    mask = torch.zeros(B, 1, L, L).masked_fill(~ok[None, None], float('-inf'))  # convert to the format the model expects: 0 = allowed, -inf = blocked
    last = model.transformer(inputs_embeds=seq, attention_mask=mask).last_hidden_state[:, -1]   # hidden state at the EOT position
    return model.lm_head(last), torch.stack(thoughts, 1)       # (answer scores (B,14), thoughts (B, N, 128))

**Line-by-line summary of the loop (`for i in range(N)`):**
1. Run all layers over everything so far.
2. Keep only the **last position's** hidden state: it summarises the prompt (and earlier thoughts).
3. (Optionally overwrite it: patching.)
4. Store it as a thought.
5. Append it to the input sequence as if it were a token's vector.

After the loop, `EOT` is appended and the answer is read at that position. This is `coconut_prefix` + `coconut_forward` in the repo; the repo adds a cache so the prompt isn't recomputed on every pass, and padding for batches of different lengths. The idea is identical.

**Why the bottleneck line matters:** `ok[-1, :5] = False` is the only difference between the two twins. With it, anything the answer knows about the question (including the hint) must travel through the thoughts.

In [ ]:
p, _, _, _ = make_data(3)                                      # three random questions
logits, thoughts = run(model, p)                               # run the (still untrained here / lightly trained in A4) model
print('answer scores shape:', tuple(logits.shape))             # (3, 14): scores for each token, for 3 questions
print('thoughts shape     :', tuple(thoughts.shape))           # (3, 2, 128): 3 questions x N=2 thoughts x 128 numbers each

**What to notice:** each question now has `N=2` thoughts, each a 128-number vector that was never turned into a word.

# Part D. Train two twins

We train two fresh models with the A4 loop, on the task from Part B, going through the Part C forward pass:
* **free twin** (`bottleneck=False`): the answer can read the prompt (and hint) directly. In the real experiment, models like this ended up not *using* their thoughts.
* **bottleneck twin** (`bottleneck=True`): the answer can only read the thoughts. A causal route through the thoughts exists **by construction**. It is the positive control.

The real code trains in stages (written steps are gradually replaced by thoughts, a 'curriculum'). A 1-step toy does not need that, so we train end to end.

In [ ]:
def train(bottleneck, steps=1500):
    m = GPT2LMHeadModel(config)                                # a fresh random model (same architecture as above)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)           # the optimizer (A4)
    for s in range(steps):                                     # repeat the A4 loop
        prompt, _, _, target = make_data(256)                  # a fresh batch of 256 questions each step
        logits, _ = run(m, prompt, bottleneck)                 # forward pass THROUGH the thoughts (Part C)
        loss = F.cross_entropy(logits, target)                 # loss on the answer token only
        opt.zero_grad(); loss.backward(); opt.step()           # clear, compute gradients, nudge (gradients flow back through the thoughts)
    return m.eval()                                            # .eval(): switch off training-only behaviour (dropout) so runs are repeatable

free, bneck = train(False), train(True)                        # train both twins (a few minutes on CPU)

**What to notice:** `loss.backward()` sends the learning signal back *through the fed-back thoughts*. That is how the model learns what to put in a thought without ever being told. Check accuracy before moving on (next cell).

In [ ]:
@torch.no_grad()                                               # no gradients needed: we are only measuring
def accuracy(model, bn):
    prompt, _, _, target = make_data(3000)                     # fresh questions
    pred = run(model, prompt, bn)[0][:, :V].argmax(1)          # predicted option (only look at the 10 option tokens)
    return (pred == target).float().mean().item()              # fraction that match what the model was trained to say

print('free  :', accuracy(free, False))                        # should be near 1.0 if training worked
print('bneck :', accuracy(bneck, True))                        # may be lower or need more steps: this is the harder twin

**What to notice:** if a twin is far below 1.0 it has not learned the task and the later parts would be meaningless. Raise `steps` in `train` and retrain. (The real experiment gates on accuracy for the same reason.)

# Part E. Counterfactual labels

How do we know the hint *caused* a wrong answer, rather than the model being wrong anyway? **Run the same question twice**: once with the wrong hint, once with no hint.

* **Harmful switch** = without the hint the model answered correctly, with the wrong hint it answered the planted wrong option.

That fact, not 'the answer looked bad', is the label that monitors are scored against.

In [ ]:
@torch.no_grad()
def answers(model, prompt, bn):
    return run(model, prompt, bn)[0][:, :V].argmax(1)          # the model's chosen option for each question

def counterfactual(model, bn, m=4000):
    prompt, true, wrong, _ = make_data(m)                      # m fresh questions
    hinted = prompt.clone(); hinted[:, 3] = wrong              # copy of the prompts with a WRONG hint in slot 3
    plain = prompt.clone();  plain[:, 3] = NONE                # copy of the same prompts with NO hint in slot 3
    ph, pu = answers(model, hinted, bn), answers(model, plain, bn)   # answer WITH hint (ph), answer WITHOUT hint (pu)
    harmful = (pu == true) & (ph == wrong)                     # right without the hint AND equal to the planted wrong option with it
    return hinted, plain, true, wrong, harmful

for name, model_, bn in (('free', free, False), ('bneck', bneck, True)):   # report for each twin
    hinted, plain, true, wrong, harmful = counterfactual(model_, bn)
    print(name, '| harmful switches:', int(harmful.sum()), 'of', len(harmful))

**What to notice:** if the count is 0 the twin never followed a wrong hint, so there is nothing to patch or probe. In the real experiment this is the 'harmful switch' count in each `RUN.md`. Mirrors `uptake_label(..., wrong=True)`.

# Part F. Activation patching

**Question:** do the thoughts carry the hint effect?

**Procedure** (on harmful-switch rows only):
1. Run the question *without* the hint and save its thoughts (the 'clean' thoughts).
2. Run the question *with* the wrong hint, but at the thought positions overwrite the model's thoughts with the clean ones (Part C's `patch`).
3. See whether the answer goes back to the true option.

If it does, the thoughts carry the cause. If not, the hint reaches the answer some other way (in the free twin: directly through attention to the prompt).

**Controls, because 'the answer changed' alone proves nothing.** Overwriting activations with *anything* can disturb the model. So we also overwrite with:
* **zeros**, and
* **another question's thoughts** (realistic-looking but wrong).

A real route restores clearly more than these controls do.

In [ ]:
@torch.no_grad()
def patching(model, bn):
    hinted, plain, true, wrong, harmful = counterfactual(model, bn)   # find the harmful-switch rows (Part E)
    if harmful.sum() == 0: return 'no harmful switches: nothing to patch'
    h, p, t = hinted[harmful], plain[harmful], true[harmful]   # keep only those rows: wrong-hint prompt, no-hint prompt, true answer
    _, clean = run(model, p, bn)                               # thoughts from the NO-HINT run: the source we copy from
    def back_to_true(patch):                                   # helper: fraction of rows that answer the TRUE option again
        pred = run(model, h, bn, patch)[0][:, :V].argmax(1)    # re-run the HINTED prompts, with the given patch applied
        return (pred == t).float().mean().item()
    all_slots = lambda x: {i: x[:, i] for i in range(N)}       # build a patch dict that overwrites every thought slot
    return dict(
        no_patch = back_to_true(None),                         # sanity: no change, so ~0 (these rows are harmful by definition)
        restore  = back_to_true(all_slots(clean)),             # THE TEST: clean thoughts written into the hinted run
        zero     = back_to_true(all_slots(torch.zeros_like(clean))),   # control 1: overwrite with zeros
        other_q  = back_to_true(all_slots(clean.roll(1, 0))),  # control 2: each row gets the NEXT row's thoughts (another question)
        one_slot = [back_to_true({i: clean[:, i]}) for i in range(N)]) # patch one thought at a time

for name, model_, bn in (('free', free, False), ('bneck', bneck, True)):
    print(name, patching(model_, bn))

**How to read the output**
* `no_patch` should be near 0.
* `restore` is the test. Compare it with `zero` and `other_q`, **not** with 0.
* The pattern the real experiment found: free twin `restore` ≈ `no_patch` (thoughts inert); bottleneck twin `restore` far above both controls (route exists), `one_slot` partial (the route is spread across thoughts).
* If the toy does not show this, that is a fact about the toy (size, steps, task), not about the method. Do not force it.

Real code: `analysis/latent_patching.py`, `run(..., patch=...)`; the controls match `docs/causal_protocol.md`.

# Part G. A probe monitor, and why baselines matter

A **monitor** reads the model's internals and predicts a behaviour. A **probe** is a monitor trained on labelled examples. Here: a simple classifier (logistic regression) on the thoughts of the hinted run, trained to predict the Part E label (harmful switch).

A probe score alone means nothing. Two comparisons:
* **Output check baseline:** 'did the answer equal the hint?' Needs no internals; on wrong-hint prompts it can already score very high.
* **Shuffled-label control:** the same probe trained on randomly shuffled labels. Should score about 0.5 (chance). If it does not, something is leaking.

The score is **AUROC**: 1.0 = perfectly separates harmful switches from the rest; 0.5 = chance.

In [ ]:
@torch.no_grad()
def probe_demo(model, bn):
    prompt, true, wrong, _ = make_data(6000)                   # fresh questions
    hinted = prompt.clone(); hinted[:, 3] = wrong              # wrong hint on every question
    plain = prompt.clone();  plain[:, 3] = NONE                # no hint
    lg, th = run(model, hinted, bn)                            # run WITH hint; keep the thoughts: the monitor's input
    ph = lg[:, :V].argmax(1); pu = answers(model, plain, bn)   # answers with and without the hint
    y = ((pu == true) & (ph == wrong)).numpy().astype(int)     # counterfactual label per question (Part E): 1 = harmful switch
    X = th.reshape(len(y), -1).numpy()                         # flatten each question's N thoughts into one long feature vector
    k = len(y) // 2                                            # first half trains the probe, second half tests it
    if len(set(y[:k])) < 2 or len(set(y[k:])) < 2: return 'too few harmful switches for a probe'
    clf = LogisticRegression(C=0.01, max_iter=2000).fit(X[:k], y[:k])        # train the probe on the first half
    auc_probe = roc_auc_score(y[k:], clf.predict_proba(X[k:])[:, 1])         # score the probe on the held-out half
    auc_output = roc_auc_score(y[k:], (ph == wrong).numpy()[k:].astype(float))   # BASELINE: 'answer equals the wrong hint'
    shuf = LogisticRegression(C=0.01, max_iter=2000).fit(X[:k], np.random.permutation(y[:k]))   # CONTROL: labels shuffled
    auc_shuf = roc_auc_score(y[k:], shuf.predict_proba(X[k:])[:, 1])         # should be near 0.5
    return dict(probe=auc_probe, output_check=auc_output, shuffled_label=auc_shuf)

for name, model_, bn in (('free', free, False), ('bneck', bneck, True)):
    print(name, probe_demo(model_, bn))

**What to notice**
* If `probe` is not clearly above `output_check`, the probe adds nothing over looking at the answer.
* A probe that beats both is **still not shown to track the cause**: it may be reading a side-effect of the hint. Part F is the test of cause; Part G is the test of prediction. The project's open question (monitor validity) is the gap between them.

Real code: `probe`, `fit_score`, `report` (which adds bootstrap confidence intervals) in `prosqa_monitor.py`.

# Part H. Logit lens

A thought is a hidden state, and the LM head (A2) turns any hidden state into word scores. So we can apply the LM head to a **thought** and see which tokens it 'points at'. This is descriptive only: a thought that decodes to the hint does not prove the model *uses* it. Pair it with patching (Part F) to say anything causal.

In [ ]:
@torch.no_grad()
def lens(model, bn, show=5):
    prompt, true, wrong, _ = make_data(show)                   # a few questions
    hinted = prompt.clone(); hinted[:, 3] = wrong              # give each a wrong hint
    _, th = run(model, hinted, bn)                             # run it and keep the thoughts, shape (show, N, 128)
    for r in range(show):                                      # for each question...
        row = [model.lm_head(th[r, i]).topk(3).indices.tolist() for i in range(N)]   # ...decode each thought: its top-3 tokens
        print(f'a={prompt[r,0].item()} b={prompt[r,1].item()} hint={wrong[r].item()} true={true[r].item()} -> thoughts decode to', row)

print('free');  lens(free, False)
print('bneck'); lens(bneck, True)

**What to notice:** do the top tokens include `a`, `b`, the hint or the true answer, or special tokens like `10` (`BOT`)? Differences between the twins are an observation to follow up with patching, not a conclusion. Token ids 10 to 13 are the special tokens.

# What is not in this notebook
Cross-format transfer, label-free detectors (Mahalanobis, activation matching), the staged curriculum, and attention-mask hint-blocking are in `prosqa_monitor/prosqa_monitor.py` and `analysis/latent_patching.py`. A good next read is `analysis/latent_patching.py`: its `run()` is Part C plus a cache, padding and extra blocking options.